In [42]:
import cv2
from cvzone.HandTrackingModule import HandDetector
import numpy as np
import time

cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 840)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 680)

detector = HandDetector(maxHands=1)

prev_time = 0

kalman = cv2.KalmanFilter(4, 2)  # 4 state (x, y, dx, dy), 2 measurements (x, y)
kalman.measurementMatrix = np.array([[1,0,0,0],
                                     [0,1,0,0]], np.float32)
kalman.transitionMatrix = np.array([[1,0,1,0],
                                    [0,1,0,1],
                                    [0,0,1,0],
                                    [0,0,0,1]], np.float32)
kalman.processNoiseCov = np.eye(4, dtype=np.float32) * 1e-3
kalman.measurementNoiseCov = np.eye(2, dtype=np.float32) * 1e-1
kalman.errorCovPost = np.eye(4, dtype=np.float32)

alpha_depth = 0.2
smoothed_depth = 0
alpha_trigger = 0.3
smoothed_trigger = 0

log_file = "event_log.txt"
triggered_point = None  # stick to landmark once triggered

while True:
    start_time = time.time()  # Start latency timing

    success, img = cap.read()
    if not success:
        continue

    img = cv2.flip(img, 1)
    h, w, _ = img.shape

    current_time = time.time()
    fps = 1 / (current_time - prev_time) if prev_time != 0 else 0
    prev_time = current_time

    horizontal_y = h // 2  # horizontal line in middle
    raw_trigger = 0
    depth_value = 0
    detection_text = "NO HAND"

    hands, img = detector.findHands(img)

    if hands:
        hand = hands[0]
        lmList = hand["lmList"]
        detection_text = "HAND DETECTED"

        z_values = [lm[2] for lm in lmList]
        depth_value = int(np.mean(z_values))
        smoothed_depth = alpha_depth * depth_value + (1 - alpha_depth) * smoothed_depth

        if triggered_point is None:
            # Find landmark closest to threshold
            closest_lm = min(lmList, key=lambda lm: abs(lm[1] - horizontal_y))
            x, y = closest_lm[0], closest_lm[1]

            # Kalman update
            predicted = kalman.predict()
            measurement = np.array([[np.float32(x)], [np.float32(y)]])
            corrected = kalman.correct(measurement)
            x_smooth, y_smooth = int(corrected[0]), int(corrected[1])

            # Check if it crosses threshold
            if y_smooth >= horizontal_y:
                raw_trigger = 1
                triggered_point = (x_smooth, y_smooth)  # stick to this point
        else:
            # Stick to previously triggered point
            x_smooth, y_smooth = triggered_point
            raw_trigger = 1

    smoothed_trigger = alpha_trigger * raw_trigger + (1 - alpha_trigger) * smoothed_trigger
    triggered = smoothed_trigger > 0.5

    if triggered and triggered_point:
        with open(log_file, "a") as f:
            f.write(f"{time.time()} - Triggered | Depth: {int(smoothed_depth)}\n")
        triggered_point = None  # optional: reset after logging once

    line_color = (0, 255, 0) if triggered else (0, 0, 255)
    cv2.line(img, (50, horizontal_y), (w-50, horizontal_y), line_color, 3)

    if hands:
        cv2.circle(img, (x_smooth, y_smooth), 10, (0, 255, 255), -1)

    bar_height = int(np.interp(smoothed_depth, [-150, 150], [0, h]))
    cv2.rectangle(img, (w-40, h-bar_height), (w-20, h), (255, 255, 0), -1)

    status_text = "TRIGGERED!" if triggered else "SAFE"
    cv2.putText(img, f"Status: {status_text}",
                (20, 40),
                cv2.FONT_HERSHEY_DUPLEX,
                0.8, line_color, 2, cv2.LINE_AA)

    cv2.putText(img, f"Detection: {detection_text}",
                (20, 75),
                cv2.FONT_HERSHEY_DUPLEX,
                0.7, (255,255,255), 2, cv2.LINE_AA)

    cv2.putText(img, f"Depth: {int(smoothed_depth)}",
                (20, 105),
                cv2.FONT_HERSHEY_DUPLEX,
                0.7, (255,255,0), 2, cv2.LINE_AA)

    cv2.putText(img, f"FPS: {int(fps)}",
                (20, 135),
                cv2.FONT_HERSHEY_DUPLEX,
                0.7, (0,255,255), 2, cv2.LINE_AA)

    end_time = time.time()
    latency_ms = (end_time - start_time) * 1000
    cv2.putText(img, f"Latency: {latency_ms:.1f} ms",
                (20, 165),
                cv2.FONT_HERSHEY_DUPLEX,
                0.7, (0, 255, 255), 2, cv2.LINE_AA)

    cv2.imshow("Horizontal Threshold Monitor", img)

    if cv2.waitKey(1) & 0xFF == ord('x'):
        break

cap.release()
cv2.destroyAllWindows()

C:\Users\kusha\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '
C:\Users\kusha\AppData\Local\Temp\ipykernel_11968\3723037933.py:72: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  x_smooth, y_smooth = int(corrected[0]), int(corrected[1])
